# Uncertainty pipeline — dry-run (Google Colab)

Clones `dev/uncertainty` from `PavelPodlesny/DiT`, sets up the environment, and runs
`run_uncertainty.py --dry-run` with a lightweight config so it finishes quickly on a Colab GPU.

See `uncertainty/README.md` for what the dry-run actually checks.

## 1. Connect to Google Drive

Dry-run outputs are written under `OUTPUT_DIR` on your Drive — set the path below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set this to wherever on your Drive you want dry-run results saved.
OUTPUT_DIR = '/content/drive/MyDrive/DiT_uncertainty/dry_run_results'

## 2. Clone `dev/uncertainty` from your fork

In [ ]:
%cd /content
!rm -rf DiT
!git clone --branch dev/uncertainty --single-branch https://github.com/PavelPodlesny/DiT.git
%cd DiT

## 3. Set up the environment

Colab already ships a working PyTorch + CUDA install, so we only add the packages this repo and
the uncertainty tool need on top of that (see `uncertainty/README.md`'s Setup section).

In [ ]:
!pip install -q timm diffusers accelerate
!pip install -q pyyaml lpips open_clip_torch matplotlib

import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 4. Write a lightweight dry-run config

One class, one base trajectory, few branch points, `K=1` (`--dry-run` only ever exercises the
`mock` identity perturbation regardless of the `perturbations` list, so `K>1` would just repeat
identical work), and few sampling steps — just enough to exercise the noise-replay hook end to
end quickly.

The model is kept as `DiT-XL/2` with `ckpt: null`: `find_model`'s auto-download (`download.py`)
only recognizes the released `DiT-XL/2` checkpoints, and `ModelAdapter`'s default checkpoint path
always resolves to a `DiT-XL-2-*` filename regardless of the configured `model` — pointing
`model` at a smaller preset (e.g. `DiT-S/8`) with `ckpt: null` would try to load XL/2 weights into
that smaller architecture and fail with a shape mismatch. "Lightweight" here means few steps/
classes/bases/branch-points, not a smaller model — the first run will download the ~2.5GB
XL/2 256x256 checkpoint.

In [ ]:
config_yaml = f"""
model: DiT-XL/2
image_size: 256
vae: mse
ckpt: null
num_classes: 1000
cfg_scale: 1.0
num_sampling_steps: 20
device: null

classes: [207]
M: 1
base_seed: 0
branch_points: [5, 10, 15]

K: 1
perturbation_seed: 1000
perturbations: []

output_dir: {OUTPUT_DIR}

dry_run_tolerances:
  latent_mse: 1.0e-6
  image_mse: 1.0e-4

metrics:
  lpips_net: alex
  clip_model: ViT-B-32
  clip_pretrained: openai
  dino_model: dino_vits16
"""

with open('dry_run_config.yaml', 'w') as f:
    f.write(config_yaml)

print(config_yaml)

## 5. Run the dry-run

In [ ]:
!python run_uncertainty.py --config dry_run_config.yaml --dry-run